# <center>Modeling
This notebook will answer the questions:   
- **Does treating each variable according to its actual measurement type improve predictive performance, even before creating new features?**
- **Can we predict final performance from demographic, family, school, behavioral and other student characteristics?**

## Outline:

1. Modeling objective:
- Regression Task =   
    - **Can we improve on the predictions previously established in the math_modeling_regression notebook a student's final grade?**   
- Method = **use alternative preprocessing strategies and the same modeling approaches to determine whether predictive performance can be improved further**

2. Data preparation
    - Feature/target separation
    - Train/test split
    - Numerical features
    - Categorical features
    - Encoding
    - Preprocessing pipeline
    - Leakage considerations

3. Regression

    - 3.1 Regression objective
    - 3.2 Target: Final Grade (G3)
    - 3.3 Baseline model
    - 3.4 Linear Regression
    - 3.5 Decision Tree Regressor
    - 3.6 Random Forest Regressor
    - 3.7 Model evaluation
    - 3.8 Model comparison
    - 3.9 Regression findings

4. Overall Modeling Conclusions

5. Limitations

6. Recommendations / Future Work

## Baseline results obtained from previous notebook:
| Model                   |    MAE ↓ |   RMSE ↓ |     R² ↑ |
| ----------------------- | -------: | -------: | -------: |
| Dummy Regressor         |     3.65 |     4.55 |    -0.01 |
| Linear Regression       |     3.40 |     4.20 |     0.14 |
| Decision Tree           |     3.61 |     4.76 |    -0.11 |
| Tuned Decision Tree     |     3.48 |     4.29 |     0.10 |
| Default Random Forest   |     3.09 |     3.88 |     0.27 |
| **Tuned Random Forest** | **3.03** | **3.80** | **0.30** |


In [11]:
# specify libraries to use
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# modeling libraries
# Baseline model
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Split the data
from sklearn.model_selection import train_test_split, GridSearchCV

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Decision Tree model
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# Random Forest model
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

In [12]:
# load the maths dataset
math_df = pd.read_csv('student-mat.csv', sep = ';')
math_df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [13]:
# Open and read the txt file
with open('student.txt', 'r', encoding='utf-8') as file:
    content = file.read()

# Print the text to the VS Code terminal
print(content)


# Attributes for both student-mat.csv (Math course) and student-por.csv (Portuguese language course) datasets:
1 school - student's school (binary: "GP" - Gabriel Pereira or "MS" - Mousinho da Silveira)
2 sex - student's sex (binary: "F" - female or "M" - male)
3 age - student's age (numeric: from 15 to 22)
4 address - student's home address type (binary: "U" - urban or "R" - rural)
5 famsize - family size (binary: "LE3" - less or equal to 3 or "GT3" - greater than 3)
6 Pstatus - parent's cohabitation status (binary: "T" - living together or "A" - apart)
7 Medu - mother's education (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
8 Fedu - father's education (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)
9 Mjob - mother's job (nominal: "teacher", "health" care related, civil "services" (e.g. administrative or police), "at_home" or 

In [14]:
# general info about the dataframe
math_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 20  higher    

## Preprocessing

In [15]:
# Define predictors and target variable
# G1 and G2 are excluded to maintain the same prediction setup as Iteration 1
X = math_df.drop(columns=['G1', 'G2', 'G3'])
y = math_df['G3']


# Define variables according to their measurement type

# Numerical variables
numerical_features = [
    'age',
    'absences'
]

# Ordinal categorical variables
ordinal_features = [
    'Medu',
    'Fedu',
    'traveltime',
    'studytime',
    'failures',
    'famrel',
    'freetime',
    'goout',
    'Dalc',
    'Walc',
    'health'
]

# Nominal categorical variables
nominal_features = [
    'school',
    'sex',
    'address',
    'famsize',
    'Pstatus',
    'Mjob',
    'Fjob',
    'reason',
    'guardian'
]

# Binary categorical variables
binary_features = [
    'schoolsup',
    'famsup',
    'paid',
    'activities',
    'nursery',
    'higher',
    'internet',
    'romantic'
]


# --------------------------------------------------
# Preprocessor for Linear Regression
# --------------------------------------------------

# Numerical and ordinal variables retain their numerical/ordinal information
# but are scaled for the linear model
linear_numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Nominal and binary categorical variables are one-hot encoded
linear_categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(
        drop='first',
        handle_unknown='ignore'
    ))
])

linear_preprocessor = ColumnTransformer(
    transformers=[
        ('num', linear_numeric_transformer, numerical_features),
        ('ordinal', linear_numeric_transformer, ordinal_features), # treat ordinal variables as numbers
        ('cat', linear_categorical_transformer, nominal_features + binary_features)
    ]
)


# --------------------------------------------------
# Preprocessor for Decision Tree and Random Forest
# --------------------------------------------------

# Tree-based models do not require feature scaling
tree_numeric_transformer = 'passthrough'

# Nominal and binary categorical variables are one-hot encoded
tree_categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(
        drop='first',
        handle_unknown='ignore'
    ))
])

tree_preprocessor = ColumnTransformer(
    transformers=[
        ('num', tree_numeric_transformer, numerical_features + ordinal_features),
        ('cat', tree_categorical_transformer, nominal_features + binary_features)
    ]
)


# --------------------------------------------------
# Train-test split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Baseline Models
### 1. Dummy Regressor - Naive baseline
**What if we completely ignored student characteristics and just predicted the average grade for everyone?**    
- It essentially predicts the training-set mean for every student
- It doesn't look at age, study time, failures, absences, parental education, etc.
- Gives a reference point for judging whether the actual regression model has learned anything useful

In [16]:
dummy_model = DummyRegressor(strategy='mean')

dummy_model.fit(X_train, y_train)

dummy_y_pred = dummy_model.predict(X_test)

# evaluation
dummy_mae = mean_absolute_error(y_test, dummy_y_pred)
dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_y_pred))
dummy_r2 = r2_score(y_test, dummy_y_pred)

print(f"Dummy MAE: {dummy_mae:.2f}")
print(f"Dummy RMSE: {dummy_rmse:.2f}")
print(f"Dummy R²: {dummy_r2:.2f}")

Dummy MAE: 3.65
Dummy RMSE: 4.55
Dummy R²: -0.01


In [17]:
linear_model = Pipeline(steps=[
    ('preprocessor', linear_preprocessor),
    ('regressor', LinearRegression())
])


# fit the model
linear_model.fit(X_train, y_train)

# make predictions
baseline_y_pred = linear_model.predict(X_test)

# evaluation
baseline_mae = mean_absolute_error(y_test, baseline_y_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_y_pred))
baseline_r2 = r2_score(y_test, baseline_y_pred)

print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Baseline R²: {baseline_r2:.2f}")

Baseline MAE: 3.40
Baseline RMSE: 4.20
Baseline R²: 0.14


### **Interpretation:**  
Treating ordinal values as numbers and passing them through the numeric transformer yielded no difference at all in regression metrics. However, this is to be expected because in the previous notebook, they were treated as numeric variables as well despite their ordinal nature.

## Take the Ordinal Variables Through OneHotEncoder

**Juestification:**

A variable like:  
 - *Medu* - mother's education: (numeric: 0 - none,  1 - primary education (4th grade), 2 – 5th to 9th grade, 3 – secondary education or 4 – higher education)

Actually represents categorical values that have an inherent order. Instead of renaming the numbers to categorical variables, treat them like categorical variables and pass them through one hot encoding.


In [18]:
# Define predictors and target variable
# G1 and G2 are excluded to maintain the same prediction setup as Iteration 1
X = math_df.drop(columns=['G1', 'G2', 'G3'])
y = math_df['G3']


# Define variables according to their measurement type

# Numerical variables
numerical_features = [
    'age',
    'absences'
]

# Ordinal categorical variables
ordinal_features = [
    'Medu',
    'Fedu',
    'traveltime',
    'studytime',
    'failures',
    'famrel',
    'freetime',
    'goout',
    'Dalc',
    'Walc',
    'health'
]

# Nominal categorical variables
nominal_features = [
    'school',
    'sex',
    'address',
    'famsize',
    'Pstatus',
    'Mjob',
    'Fjob',
    'reason',
    'guardian'
]

# Binary categorical variables
binary_features = [
    'schoolsup',
    'famsup',
    'paid',
    'activities',
    'nursery',
    'higher',
    'internet',
    'romantic'
]


# --------------------------------------------------
# Preprocessor for Linear Regression
# --------------------------------------------------

# Numerical variables are scaled
linear_numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Ordinal, nominal, and binary variables are one-hot encoded
# This treats each category as a separate level rather than
# assuming equal numerical distances between ordinal levels
linear_categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(
        drop='first',
        handle_unknown='ignore'
    ))
])

linear_preprocessor2 = ColumnTransformer(
    transformers=[
        ('num', linear_numeric_transformer, numerical_features),
        ('ordinal', linear_categorical_transformer, ordinal_features),
        ('cat', linear_categorical_transformer, nominal_features + binary_features)
    ]
)


# --------------------------------------------------
# Preprocessor for Decision Tree and Random Forest
# --------------------------------------------------

# Tree-based models do not require feature scaling
tree_numeric_transformer = 'passthrough'

# Ordinal, nominal, and binary variables are one-hot encoded
tree_categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(
        drop='first',
        handle_unknown='ignore'
    ))
])

tree_preprocessor2 = ColumnTransformer(
    transformers=[
        ('num', tree_numeric_transformer, numerical_features),
        ('ordinal', tree_categorical_transformer, ordinal_features),
        ('cat', tree_categorical_transformer, nominal_features + binary_features)
    ]
)


# --------------------------------------------------
# Train-test split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [19]:
dummy_model = DummyRegressor(strategy='mean')

dummy_model.fit(X_train, y_train)

dummy_y_pred = dummy_model.predict(X_test)

# evaluation
dummy_mae = mean_absolute_error(y_test, dummy_y_pred)
dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_y_pred))
dummy_r2 = r2_score(y_test, dummy_y_pred)

print(f"Dummy MAE: {dummy_mae:.2f}")
print(f"Dummy RMSE: {dummy_rmse:.2f}")
print(f"Dummy R²: {dummy_r2:.2f}")

Dummy MAE: 3.65
Dummy RMSE: 4.55
Dummy R²: -0.01


In [20]:
linear_model = Pipeline(steps=[
    ('preprocessor', linear_preprocessor2),
    ('regressor', LinearRegression())
])


# fit the model
linear_model.fit(X_train, y_train)

# make predictions
baseline_y_pred = linear_model.predict(X_test)

# evaluation
baseline_mae = mean_absolute_error(y_test, baseline_y_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_y_pred))
baseline_r2 = r2_score(y_test, baseline_y_pred)

print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Baseline R²: {baseline_r2:.2f}")

Baseline MAE: 3.66
Baseline RMSE: 4.79
Baseline R²: -0.12


**Interpretation:**  
Despite the changes so far, i.e separating ordinal, nominal, categorical, numeric values, and either treating ordinal values as numeric or categorical, there is no significant change in model performance. 
The dummy regressor yielded the exact values as in the previous notebook, however, baseline metrics changed. 

Previous metrics:
- Baseline MAE: 3.40
- Baseline RMSE: 4.20
- Baseline R²: 0.14

Current metrics:
- Baseline MAE: 3.66
- Baseline RMSE: 4.79
- Baseline R²: -0.12

There is an increase in the errors, and the model now explains even less of the variation in the outcome variable. This means that treating ordinal values as categorical and onehot encoding them may be incorrect as it results in a worse baseline model using linear regression.  
Next steps would be to confirm this behavior using decision tree and random forest algorithms.

## Decision Tree Regressor
### Default Model

In [21]:
# model the data - ignore the first iteration because it is similar to previous notebook work
tree_model = Pipeline(steps=[
    ('preprocessor', tree_preprocessor2),
    ('regressor', DecisionTreeRegressor()) # G3 is a numeric final grade (0–20)
])

# fit the model
tree_model.fit(X_train, y_train)

# make predictions
tree_y_pred = tree_model.predict(X_test)

# evaluation
tree_mae = mean_absolute_error(y_test, tree_y_pred)
tree_rmse = np.sqrt(mean_squared_error(y_test, tree_y_pred))
tree_r2 = r2_score(y_test, tree_y_pred)

print(f"Decision Tree MAE: {tree_mae:.2f}")
print(f"Decision Tree RMSE: {tree_rmse:.2f}")
print(f"Decision Tree R²: {tree_r2:.2f}")

Decision Tree MAE: 4.19
Decision Tree RMSE: 5.76
Decision Tree R²: -0.62


**Interpretation:** Default decision tree performs worse on all metrics.  
Previous metrics:  
- Decision Tree MAE: 3.61
- Decision Tree RMSE: 4.79
- Decision Tree R²: -0.12



### Modeling A Controlled/Tuned Decision Tree

In [22]:
# creating a parameter grid
param_grid = {
    'regressor__max_depth': [3,4,5,6,8,10], # note that there is a double underscore after the word 'regressor'
    'regressor__min_samples_split': [2,5,10,20], # because the pipeline already has a 'preprocessor' and 'regressor', the double underscore means...
    'regressor__min_samples_leaf': [1,2,4,8] # "Access a parameter belonging to the regressor inside the Pipeline."
} # therefore, 'regressor__max_depth' means the max_depth parameter of the Decision Tree.

# create the gridsearchcv object
grid_search = GridSearchCV(
    estimator=tree_model,
    param_grid=param_grid,
    cv=5, # 5-fold cross-validation
    scoring='r2', # Select the model with the highest mean cross-validation R²
    n_jobs=-1 # tells scikit-learn to use all available CPU cores to speed up the search.
)

# fit the gridsearchcv
grid_search.fit(X_train, y_train)

# finding the best parameters
print("Best parameters:")
print(grid_search.best_params_)

# print(f"Best parameters: {grid_search.best_params_}")

# finding best cross validation score
print("Best cross_validation R^2:")
print(grid_search.best_score_)

Best parameters:
{'regressor__max_depth': 3, 'regressor__min_samples_leaf': 8, 'regressor__min_samples_split': 2}
Best cross_validation R^2:
0.008539923234584256


In [23]:
# Make predictions using the best model
tree_grid_y_pred = grid_search.predict(X_test)

# Evaluate the tuned Decision Tree
tree_grid_mae = mean_absolute_error(y_test, tree_grid_y_pred)
tree_grid_rmse = np.sqrt(mean_squared_error(y_test, tree_grid_y_pred))
tree_grid_r2 = r2_score(y_test, tree_grid_y_pred)

print(f"Tuned Decision Tree MAE: {tree_grid_mae:.2f}")
print(f"Tuned Decision Tree RMSE: {tree_grid_rmse:.2f}")
print(f"Tuned Decision Tree R²: {tree_grid_r2:.2f}")

Tuned Decision Tree MAE: 3.57
Tuned Decision Tree RMSE: 4.30
Tuned Decision Tree R²: 0.10


## Conclusion

Treating ordinal variables as categorical decreased the performance of the modeling algorithms employed in this notebook.